# Генерація тестових даних
Ноутбук для наповнення БД синтетичними даними, згенерованими за допомогою LLM.

## 1. Імпорти та конфігурація

In [9]:
import os, json, sys
os.chdir(r"E:/projects/University/BKR/bcr-app/packages/backend")
sys.path.insert(0, '..')

## 2. Підключення до БД

In [10]:
import nest_asyncio
import asyncio
from src.db.session import get_async_sessionmaker

In [11]:
nest_asyncio.apply()

session_factory = get_async_sessionmaker()

In [12]:
from sqlalchemy import delete
from src.db.models import Entry, Goal, Report, User

async def clear_db():
    async with session_factory() as session:
        await session.execute(delete(Entry))
        await session.execute(delete(Report))
        await session.execute(delete(Goal))
        await session.execute(delete(User))
        await session.commit()
        print("DB cleared!")

asyncio.run(clear_db())

2026-04-24 20:27:48,052 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-24 20:27:48,054 INFO sqlalchemy.engine.Engine DELETE FROM entry
2026-04-24 20:27:48,055 INFO sqlalchemy.engine.Engine [cached since 195.3s ago] {}
2026-04-24 20:27:48,060 INFO sqlalchemy.engine.Engine DELETE FROM report
2026-04-24 20:27:48,060 INFO sqlalchemy.engine.Engine [cached since 195.2s ago] {}
2026-04-24 20:27:48,063 INFO sqlalchemy.engine.Engine DELETE FROM goal
2026-04-24 20:27:48,065 INFO sqlalchemy.engine.Engine [cached since 195.2s ago] {}
2026-04-24 20:27:48,065 INFO sqlalchemy.engine.Engine DELETE FROM "user"
2026-04-24 20:27:48,065 INFO sqlalchemy.engine.Engine [cached since 195.2s ago] {}
2026-04-24 20:27:48,070 INFO sqlalchemy.engine.Engine COMMIT
DB cleared!


## 3. Генерація даних через LLM

In [13]:
from pydantic import BaseModel
from enum import Enum
from datetime import date
from langchain_core.messages import SystemMessage

In [14]:
class Status(Enum):
    ACTIVE = "active"
    FINISHED = "finished"
    POSTPONE = "postpone"

In [15]:
# --- Моделі для генерації структури (без нотаток) ---

class GoalSeedNoNotes(BaseModel):
    title: str
    description: str | None
    status: Status
    deadline: date | None
    created_at: date
    activity_end: date  # остання дата активності по цілі

class UserSeedNoNotes(BaseModel):
    name: str
    email: str
    language: str
    goals: list[GoalSeedNoNotes]

class SeedStructure(BaseModel):
    users: list[UserSeedNoNotes]


# --- Моделі для фінального результату ---

class Note(BaseModel):
    date_note: date
    note: str
    productivity_score: int

class GoalSeed(BaseModel):
    title: str
    description: str | None
    status: Status
    deadline: date | None
    created_at: date
    notes: list[Note]

class UserSeed(BaseModel):
    name: str
    email: str
    language: str
    goals: list[GoalSeed]

class SeedData(BaseModel):
    users: list[UserSeed]

class NotesOnly(BaseModel):
    notes: list[Note]

In [16]:
from functools import lru_cache
from langchain_openai import ChatOpenAI
from src.core.settings import get_settings

@lru_cache(maxsize=1)
def get_llm() -> ChatOpenAI:
    settings = get_settings()
    return ChatOpenAI(
        model=settings.llm_model,
        api_key=settings.openai_api_key,
    )

llm = get_llm()

### Крок 1 — Структура користувачів і цілей

In [17]:
STRUCTURE_PROMPT = """Generate user and goal structure for a goal-tracking app. NO entries needed.

### Users
5 distinct users with different personalities, lifestyles and goal types.
- User 1: Ukrainian (language: "Ukrainian")
- User 2: English (language: "English")
- User 3: Ukrainian (language: "Ukrainian")
- User 4: English (language: "English")
- User 5: Ukrainian (language: "Ukrainian")

### Time frame
The tracking period is April 2025 – April 2026 (one non-calendar year).
- created_at: between 2025-04-01 and 2025-07-01 (goals start early in the year)
- activity_end: the last date the user logged anything for this goal
  - For `active` goals: set activity_end close to 2026-04-30 (goal is still ongoing)
  - For `finished` or `postpone` goals: activity_end can be anywhere from 2025-09-01 to 2026-04-30

### Goals
Each user should have 2-3 goals. Most goals should span 6-12 months — these are serious long-term commitments, not quick tasks.

Goal variety across all users:
- Types: sport, learning, career, health, creative hobby
- Statuses must include across all users:
  - At least two `finished` goals (completed successfully before the period ends)
  - At least two `postpone` goals (started but eventually put on hold)
  - Rest are `active` (still in progress at the end of the period)

For each goal set realistic created_at and activity_end that reflect how long the user actually worked on it.
Respond with valid JSON only.
"""

In [18]:
structure_llm = llm.with_structured_output(SeedStructure)

structure: SeedStructure = await asyncio.to_thread(
    structure_llm.invoke,
    [SystemMessage(content=STRUCTURE_PROMPT)]
)

for user in structure.users:
    print(f"\n{user.name} ({user.language}):")
    for goal in user.goals:
        print(f"  [{goal.status.value}] {goal.title}: {goal.created_at} → {goal.activity_end}")


Олена Коваленко (Ukrainian):
  [active] Повернути регулярний біг і підготуватися до півмарафону: 2025-04-10 → 2026-04-28
  [finished] Вивчити англійську до рівня B2: 2025-04-18 → 2025-12-18

Michael Harris (English):
  [active] Complete a full-stack portfolio project: 2025-05-02 → 2026-04-25
  [postpone] Run a consistent 5K training plan: 2025-04-22 → 2025-10-12
  [active] Improve sleep and daily recovery habits: 2025-06-05 → 2026-04-29

Ірина Мельник (Ukrainian):
  [active] Запустити персональний блог про ілюстрацію: 2025-05-14 → 2026-04-27
  [finished] Опанувати цифровий скетчинг на планшеті: 2025-04-07 → 2025-12-22

Sarah Bennett (English):
  [finished] Earn a project management certification: 2025-04-15 → 2025-10-20
  [active] Build a strength training routine: 2025-06-01 → 2026-04-30

Андрій Шевченко (Ukrainian):
  [postpone] Перейти на здорове харчування та стабільний режим сну: 2025-04-25 → 2025-11-08
  [active] Підготуватися до переїзду в нове місто: 2025-07-01 → 2026-04-26
  

e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SeedStructure(users=[User...e.date(2026, 4, 24))])]), input_type=SeedStructure])
  return self.__pydantic_serializer__.to_python(


### Крок 2 — Генерація записів по кожній цілі

In [19]:
ENTRIES_PROMPT = """Generate journal entries for one goal in a personal tracking app.

User: {name}
Language: {language}
Goal: {title}
Description: {description}
Status: {status}
Period: {start} to {end} ({months} months)

REQUIREMENTS:
- Generate EXACTLY {count} entries
- Spread evenly across the full period: ~15 entries per month
- Write all notes in the user's language ({language})
- Progress arc: early motivation → struggles → adaptation → (resolution if finished/postpone)
- productivity_score varies naturally 1-5, reflecting real highs and lows
- Notes: specific, personal, first person — mention real actions, emotions, obstacles
- Date gaps are normal, not every day
- All dates must fall between {start} and {end}

Respond with valid JSON only.
"""

notes_llm = llm.with_structured_output(NotesOnly)

async def generate_notes(user: UserSeedNoNotes, goal: GoalSeedNoNotes) -> list[Note]:
    months = max(1, (goal.activity_end.year - goal.created_at.year) * 12
                 + goal.activity_end.month - goal.created_at.month + 1)
    count = months * 15

    prompt = ENTRIES_PROMPT.format(
        name=user.name,
        language=user.language,
        title=goal.title,
        description=goal.description,
        status=goal.status.value,
        start=goal.created_at,
        end=goal.activity_end,
        months=months,
        count=count,
    )
    result = await asyncio.to_thread(
        notes_llm.invoke,
        [SystemMessage(content=prompt)]
    )
    return result.notes

In [20]:
full_users = []

for user in structure.users:
    goals_with_notes = []
    for goal in user.goals:
        print(f"Generating: {user.name} — {goal.title}...")
        notes = await generate_notes(user, goal)
        print(f"  → {len(notes)} entries")

        goals_with_notes.append(GoalSeed(
            title=goal.title,
            description=goal.description,
            status=goal.status,
            deadline=goal.deadline,
            created_at=goal.created_at,
            notes=notes,
        ))

    full_users.append(UserSeed(
        name=user.name,
        email=user.email,
        language=user.language,
        goals=goals_with_notes,
    ))

result = SeedData(users=full_users)

print("\n=== Summary ===")
for user in result.users:
    print(f"\n{user.name}:")
    for goal in user.goals:
        print(f"  [{goal.status.value}] {goal.title}: {len(goal.notes)} entries")

Generating: Олена Коваленко — Повернути регулярний біг і підготуватися до півмарафону...


e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=NotesOnly(notes=[Note(dat... productivity_score=5)]), input_type=NotesOnly])
  return self.__pydantic_serializer__.to_python(


  → 102 entries
Generating: Олена Коваленко — Вивчити англійську до рівня B2...


e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=NotesOnly(notes=[Note(dat... productivity_score=4)]), input_type=NotesOnly])
  return self.__pydantic_serializer__.to_python(


  → 161 entries
Generating: Michael Harris — Complete a full-stack portfolio project...
  → 119 entries
Generating: Michael Harris — Run a consistent 5K training plan...


e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=NotesOnly(notes=[Note(dat... productivity_score=3)]), input_type=NotesOnly])
  return self.__pydantic_serializer__.to_python(


  → 75 entries
Generating: Michael Harris — Improve sleep and daily recovery habits...
  → 328 entries
Generating: Ірина Мельник — Запустити персональний блог про ілюстрацію...
  → 177 entries
Generating: Ірина Мельник — Опанувати цифровий скетчинг на планшеті...
  → 156 entries
Generating: Sarah Bennett — Earn a project management certification...
  → 76 entries
Generating: Sarah Bennett — Build a strength training routine...
  → 111 entries
Generating: Андрій Шевченко — Перейти на здорове харчування та стабільний режим сну...
  → 143 entries
Generating: Андрій Шевченко — Підготуватися до переїзду в нове місто...
  → 139 entries
Generating: Андрій Шевченко — Навчитися грати на гітарі для власного задоволення...
  → 93 entries

=== Summary ===

Олена Коваленко:
  [active] Повернути регулярний біг і підготуватися до півмарафону: 102 entries
  [finished] Вивчити англійську до рівня B2: 161 entries

Michael Harris:
  [active] Complete a full-stack portfolio project: 119 entries
  [postpon

In [21]:
from IPython.display import HTML
import json

data = json.dumps(result.model_dump(mode="json"), indent=2, ensure_ascii=False)
display(HTML(f'<pre style="height:500px;overflow:auto">{data}</pre>'))

## 4. Ембединги

In [22]:
from sentence_transformers import SentenceTransformer

In [23]:
embedding_model = SentenceTransformer('Alibaba-NLP/gte-multilingual-base', trust_remote_code=True)

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## 5. Запис у БД

In [24]:
from src.db.models import Goal, User, Entry


async def seed():
    async with session_factory() as session:
        for user_data in result.users:
            user = User(
                name=user_data.name,
                email=user_data.email,
                language=user_data.language,
            )
            session.add(user)
            await session.flush()

            for goal_data in user_data.goals:
                goal = Goal(
                    user_id=user.id,
                    title=goal_data.title,
                    description=goal_data.description,
                    status=goal_data.status.value,
                    created_at=goal_data.created_at,
                    deadline=goal_data.deadline,
                )
                session.add(goal)
                await session.flush()

                for entry_data in goal_data.notes:
                    emb = embedding_model.encode(entry_data.note).tolist()

                    entry = Entry(
                        goal_id=goal.id,
                        date_note=entry_data.date_note,
                        note=entry_data.note,
                        productivity_score=entry_data.productivity_score,
                        embedding=emb,
                    )
                    session.add(entry)

        await session.commit()
        print("Done!")

asyncio.run(seed())

2026-04-24 20:39:12,336 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-24 20:39:12,342 INFO sqlalchemy.engine.Engine INSERT INTO "user" (name, email, language, gender) VALUES (%(name)s::VARCHAR, %(email)s::VARCHAR, %(language)s::VARCHAR, %(gender)s) RETURNING "user".id
2026-04-24 20:39:12,342 INFO sqlalchemy.engine.Engine [generated in 0.00227s] {'name': 'Олена Коваленко', 'email': 'olena.kovalenko@example.com', 'language': 'Ukrainian', 'gender': None}
2026-04-24 20:39:12,377 INFO sqlalchemy.engine.Engine INSERT INTO goal (user_id, title, description, deadline, created_at, status) VALUES (%(user_id)s::INTEGER, %(title)s::VARCHAR, %(description)s::VARCHAR, %(deadline)s::DATE, %(created_at)s::DATE, %(status)s) RETURNING goal.id
2026-04-24 20:39:12,379 INFO sqlalchemy.engine.Engine [generated in 0.00133s] {'user_id': 21, 'title': 'Повернути регулярний біг і підготуватися до півмарафону', 'description': 'Поступово відновити бігову форму, збільшити дистанцію та вийти на стабільні тр

## 6. Верифікація

In [25]:
from sqlalchemy import func, select

async def verify():
    async with session_factory() as session:
        for model, label in [(User, 'users'), (Goal, 'goals'), (Entry, 'entries')]:
            count = await session.scalar(select(func.count()).select_from(model))
            print(f"{label}: {count}")

asyncio.run(verify())

2026-04-24 20:44:46,085 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-24 20:44:46,089 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM "user"
2026-04-24 20:44:46,090 INFO sqlalchemy.engine.Engine [generated in 0.00159s] {}
users: 5
2026-04-24 20:44:46,131 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM goal
2026-04-24 20:44:46,132 INFO sqlalchemy.engine.Engine [generated in 0.00130s] {}
goals: 12
2026-04-24 20:44:46,137 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM entry
2026-04-24 20:44:46,139 INFO sqlalchemy.engine.Engine [generated in 0.00115s] {}
entries: 1680
2026-04-24 20:44:46,145 INFO sqlalchemy.engine.Engine ROLLBACK


## 7. Re-embedding (запускати якщо entries вже в БД але embedding = NULL)

In [26]:
from sqlalchemy import select
from src.db.models import Entry

BATCH_SIZE = 64

async def reembed_entries():
    async with session_factory() as session:
        result = await session.execute(
            select(Entry).where(Entry.embedding == None)
        )
        entries = result.scalars().all()
        print(f"Entries without embedding: {len(entries)}")

        for i in range(0, len(entries), BATCH_SIZE):
            batch = entries[i : i + BATCH_SIZE]
            texts = [e.note for e in batch]
            embeddings = embedding_model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=False)
            for entry, emb in zip(batch, embeddings):
                entry.embedding = emb.tolist()
            await session.commit()
            print(f"  committed {min(i + BATCH_SIZE, len(entries))}/{len(entries)}")

    print("Re-embedding done!")

asyncio.run(reembed_entries())

2026-04-24 20:44:46,164 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-24 20:44:46,168 INFO sqlalchemy.engine.Engine SELECT entry.id, entry.goal_id, entry.date_note, entry.note, entry.productivity_score, entry.embedding 
FROM entry 
WHERE entry.embedding IS NULL
2026-04-24 20:44:46,170 INFO sqlalchemy.engine.Engine [generated in 0.00193s] {}
Entries without embedding: 0
2026-04-24 20:44:46,177 INFO sqlalchemy.engine.Engine ROLLBACK
Re-embedding done!
